# DiceDial: Google Colab & Remote GPU Execution Notebook

This notebook automates environment setup, dependency installation, simulation smoke testing, **RSL-RL PPO on-GPU training with Automatic Curriculum Learning (ACL)**, checkpoint resuming, evaluation, and video rendering for **DiceDial** on Google Colab or remote GPU instances.

## 1. Hardware & System Diagnostic
Verify that an NVIDIA GPU (preferably T4, A100, or L4) is active in your runtime.

In [ ]:
!nvidia-smi

## 2. (Optional) Mount Google Drive for Checkpoint Persistence
Mount Google Drive so training checkpoints, logs, and rendered videos persist across Colab session disconnections.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Workspace Setup & Repository Sync
Clone or update the DiceDial repository and enter the working directory.

In [ ]:
import os
if not os.path.exists('DiceDial'):
    !git clone https://github.com/dheerajdhillon/DiceDial.git
%cd DiceDial
!git pull

## 4. Dependency & Simulator Installation
Install PyTorch with CUDA acceleration, NVIDIA Isaac Sim base package, Isaac Lab framework, `rsl-rl`, and DiceDial in editable mode.

In [ ]:
# 4.1 PyTorch & Isaac Sim base packages
!pip install -U torch torchvision --index-url https://download.pytorch.org/whl/cu128
!pip install isaacsim --extra-index-url https://pypi.nvidia.com

In [ ]:
# 4.2 Isaac Lab Framework
import os
if not os.path.exists('../IsaacLab'):
    %cd ..
    !git clone --depth 1 https://github.com/isaac-sim/IsaacLab.git
    %cd IsaacLab
    !./isaaclab.sh --install
    %cd ../DiceDial
else:
    print('IsaacLab directory already present.')

In [ ]:
# 4.3 Install RSL-RL and DiceDial package
!pip install rsl-rl
!pip install -e ".[video,test]"

## 5. Verification & Smoke Testing
Verify face geometry math and run a quick headless simulation step test.

In [ ]:
# Run pure PyTorch geometry unit tests
!pytest

In [ ]:
# Headless simulation smoke test (171-dimensional observation verification)
!python scripts/smoke_test.py --task DiceDial-Shadow-Sequence-v0 --num_envs 16 --steps 200 --headless

## 6. Training with RSL-RL & Automatic Curriculum Learning (ACL)

Training uses a **single continuous policy run** on `DiceDial-Shadow-Sequence-v0` powered by **RSL-RL on-GPU PPO** (`num_steps_per_env=128`).

The **ACL Manager** automatically tightens the target face success criteria (`30° -> 24° -> 20° -> 16°`) as the rolling commands-per-episode mean increases, eliminating manual stage transitions.

In [ ]:
# 6.1 Short verification run (500 iterations)
!python scripts/train_rsl.py \
  --task DiceDial-Shadow-Sequence-v0 \
  --num_envs 512 \
  --max_iterations 500 \
  --run_name colab_smoke_test \
  --headless

In [ ]:
# 6.2 Full Ambitious Training Run (50,000 iterations ~ 100M steps)
!python scripts/train_rsl.py \
  --task DiceDial-Shadow-Sequence-v0 \
  --num_envs 2048 \
  --max_iterations 50000 \
  --run_name colab_strong_run \
  --headless

In [ ]:
# Launch TensorBoard inside Colab to monitor alignment, commands/episode, and ACL progress
%load_ext tensorboard
%tensorboard --logdir outputs

## 7. Resuming Training
Resume training seamlessly from any `.pt` checkpoint. The ACL state is automatically restored from the matching `.acl.json` file.

In [ ]:
# Resume training from an existing RSL-RL checkpoint
!python scripts/train_rsl.py \
  --task DiceDial-Shadow-Sequence-v0 \
  --resume outputs/DiceDial-Shadow-Sequence-v0/colab_strong_run/model_final.pt \
  --num_envs 2048 \
  --max_iterations 100000 \
  --run_name colab_strong_run_continued \
  --headless

## 8. Evaluation & Metrics Plotting
Evaluate held-out seeds and test robustness under die mass and friction variation (`DiceDial-Shadow-Robust-v0`).

In [ ]:
# Plot compact metrics trace
!python scripts/plot_metrics.py \
  --csv outputs/DiceDial-Shadow-Sequence-v0/colab_strong_run/task_metrics.csv \
  --output outputs/dicedial_training_metrics.png

In [ ]:
# Optional SB3 PPO evaluation fallback
!python scripts/evaluate.py \
  --task DiceDial-Shadow-Sequence-v0 \
  --model outputs/DiceDial-Shadow-Sequence-v0/stage3_sequence/model.zip \
  --vecnormalize outputs/DiceDial-Shadow-Sequence-v0/stage3_sequence/model_vecnormalize.pkl \
  --episodes 500 \
  --num_envs 256 \
  --seed 2026 \
  --output evaluation/colab_eval \
  --headless

## 9. Demonstration Video Generation & Overlay
Render the 6-face sequence (`1 -> 6 -> 3 -> 5 -> 2 -> 4`) and overlay real-time task metrics.

In [ ]:
# Render raw camera recording on Play env
!python scripts/play.py \
  --task DiceDial-Shadow-Play-v0 \
  --model outputs/DiceDial-Shadow-Sequence-v0/stage3_sequence/model.zip \
  --vecnormalize outputs/DiceDial-Shadow-Sequence-v0/stage3_sequence/model_vecnormalize.pkl \
  --output videos/colab_play

In [ ]:
# Annotate video with overlay telemetry
!python scripts/annotate_video.py \
  --video videos/colab_play/raw/dicedial-episode-0.mp4 \
  --metrics videos/colab_play/video_metrics.csv \
  --output videos/colab_play/dicedial_annotated.mp4